In [1]:
from typing import Optional

class Hash:
    def __init__(self, size):
        self.min_size = size
        self.size = size
        self.container = [None] * self.size
        
    def _generate_hash_key(self, key):
        h = 0
        for i in key:
            h += ord(i)
        return h%self.size
    
    def check_load_factor(self):
        return self._load_factor()
    
    def _load_factor(self):
        return self.get_count()/self.size

    def increase(self):
        self.size = (self.size * 2)
        self.container = [None] * self.size
        print(f"Increased container size to {self.size}!")

    def decrease(self):
        if self.size > self.min_size:
            self.size = self.size//2
            self.container = [None] * self.size
            print(f"Decreased container size to {self.size}!")
    
    def __setitem__(self, key, value, insert_status=False, load_adjust: Optional[bool]=False):
        if not self.is_full(validate=False):
            hash_key = self._generate_hash_key(key)
            if self.container[hash_key]:
                if isinstance(self.container[hash_key], tuple):
                    if len(self.container[hash_key]) == 2:
                        if self.container[hash_key][0] == key:
                            if not load_adjust:
                                print(f"Item {self.container[hash_key]} is updated with {(key,value)} at {hash_key}!")
                            self.container[hash_key] = (key,value)
                            insert_status = True
                   
            if not insert_status:     
                for index, data in enumerate(self.container):
                    if isinstance(data, tuple):
                        if len(data) == 2:
                            if data[0] == key:
                                if not load_adjust:
                                    print(f"Item {self.container[index]} is updated with {(key,value)} at {index} via linear probing!")
                                self.container[index] = (key,value)
                                insert_status = True
                                break
            
            if not insert_status:
                if not self.container[hash_key]:
                    if not load_adjust:
                        print(f"Inserted item: {(key,value)} at {hash_key}!")
                    self.container[hash_key] = (key,value)
                    insert_status = True
                    
            if not insert_status:
                for index in range(len(self.container)):
                    if not self.container[index]:
                        if not load_adjust:
                            print(f"Inserted item: {(key,value)} at {index} via linear probing!")
                        self.container[index] = (key,value)
                        insert_status = True
                        break
                    
            if insert_status:
                if self._load_factor() >= 0.8:
                    data = self.container
                    print(data)
                    self.increase()
                    for val in data:
                        if val:
                            k,v = val
                            self.__setitem__(k, v, load_adjust=True)
                return    
                
    def __getitem__(self, key: str):
        if not self.is_empty(validate=False):
            hash_key = self._generate_hash_key(key)
            if self.container[hash_key]:
                if isinstance(self.container[hash_key], tuple):
                    if len(self.container[hash_key]) == 2:
                        if self.container[hash_key][0] == key:
                            print(f"Item found: {self.container[hash_key]} at {hash_key}")
                            return self.container[hash_key]
                        
            for index, data in enumerate(self.container):
                if isinstance(data, tuple):
                    if len(data) == 2:
                        if data[0] == key:
                            print(f"Item found: {data} at {index} via probing!")
                            return data                       
            
            return f"No match found for {key}!"

    def __delitem__(self, key: str, delete_status=False, load_adjust: Optional[bool]=False):
        if not self.is_empty(validate=False):
            hash_key = self._generate_hash_key(key)
            if self.container[hash_key]:
                if isinstance(self.container[hash_key], tuple):
                    if len(self.container[hash_key]) == 2:
                        if self.container[hash_key][0] == key:
                            print(f"Item deleted: {self.container[hash_key]} at {hash_key}")
                            self.container[hash_key] = None
                            delete_status = True
                
            if not delete_status:            
                for index, data in enumerate(self.container):
                    if isinstance(data, tuple):
                        if len(data) == 2:
                            if data[0] == key:
                                print(f"Item deleted: {data} at {index} via probing!")
                                self.container[index] = None 
                                delete_status = True
                                break
            
            if delete_status:
                if self._load_factor() < 0.4:
                    data = self.container
                    self.decrease()
                    for val in data:
                        if val:
                            k,v = val
                            self.__setitem__(k, v, load_adjust=True)
                    return
            
            if not delete_status:
                print(f"No match found for {key} to delete!")               
    
    def is_empty(self, validate: Optional[bool]=True):
        if self.get_count() == 0:
            if not validate:
                print("HashMap is empty!")
                return
            return True
        return False

    def is_full(self, validate: Optional[bool]=True):
        if self.get_count() == self.size:
            if not validate:
                print("HashMap is full!")
                return
            return True
        return False

    def get_count(self):
        count = 0
        for item in self.container:
            if item:
                count += 1
        return count

In [2]:
if __name__ == "__main__":
    h = Hash(5)
    h['march 6'] = 10
    h['march 9'] = 20
    h['march 19'] = 30
    h['march 16'] = 40
    h['march 17'] = 50
    print(h.container)
    h['march 1'] = 90
    print(h.container)

Inserted item: ('march 6', 10) at 4!
Inserted item: ('march 9', 20) at 2!
Inserted item: ('march 19', 30) at 1!
Inserted item: ('march 16', 40) at 3!
[None, ('march 19', 30), ('march 9', 20), ('march 16', 40), ('march 6', 10)]
Increased container size to 10!
Inserted item: ('march 17', 50) at 0 via linear probing!
[('march 17', 50), ('march 19', 30), ('march 9', 20), None, None, None, None, None, ('march 16', 40), ('march 6', 10)]
Inserted item: ('march 1', 90) at 4!
[('march 17', 50), ('march 19', 30), ('march 9', 20), None, ('march 1', 90), None, None, None, ('march 16', 40), ('march 6', 10)]


In [3]:
print(h["march 17"])
print(h["march 19"])
print(h["march 6"])
print(h["march 16"])
print(h["march 99"])
print(h.container)

Item found: ('march 17', 50) at 0 via probing!
('march 17', 50)
Item found: ('march 19', 30) at 1
('march 19', 30)
Item found: ('march 6', 10) at 9
('march 6', 10)
Item found: ('march 16', 40) at 8
('march 16', 40)
No match found for march 99!
[('march 17', 50), ('march 19', 30), ('march 9', 20), None, ('march 1', 90), None, None, None, ('march 16', 40), ('march 6', 10)]


In [4]:
del h["march 17"]
del h["march 9"]
del h["march 9"]
del h["march 99"]
print(h.container)

Item deleted: ('march 17', 50) at 0 via probing!
Item deleted: ('march 9', 20) at 2
No match found for march 9 to delete!
No match found for march 99 to delete!
[None, ('march 19', 30), None, None, ('march 1', 90), None, None, None, ('march 16', 40), ('march 6', 10)]


In [5]:
print(h.is_empty())
print(h.is_full())
print(h.get_count())

False
False
4


In [6]:
del h["march 6"]
print(h.container)

Item deleted: ('march 6', 10) at 9
Decreased container size to 5!
[None, ('march 19', 30), None, ('march 16', 40), ('march 1', 90)]


In [7]:
print(h.is_empty())
print(h.is_full())
print(h.get_count())

False
False
3


In [8]:
del h["march 16"]
del h["march 19"]
del h["march 1"]

Item deleted: ('march 16', 40) at 3
Item deleted: ('march 19', 30) at 1
Item deleted: ('march 1', 90) at 4


In [9]:
print(h["march 17"])
print(h["march 19"])
print(h["march 6"])

HashMap is empty!
No match found for march 17!
HashMap is empty!
No match found for march 19!
HashMap is empty!
No match found for march 6!


In [10]:
print(h.container)

[None, None, None, None, None]
